# AI-Native ERP for a Small Trading Organization
**Real data:** Northwind trading company database from GitHub (`jpwhite3/northwind-SQLite3`, MIT licence) — downloaded at runtime, nothing is synthesized.
**Free stack only:** Google Colab (free) · Gemini API free tier and/or Groq free tier · local `sentence-transformers` embeddings · SQLite.
**AI everywhere:** RAG knowledge base · Text-to-SQL agent with self-repair · forecasting / anomaly / segmentation · procurement agent with human approval · **AI Dev Studio** that writes, tests and registers new ERP modules automatically.

**How to run:** `Runtime > Run all`. When asked, paste a free API key (Gemini: https://aistudio.google.com/apikey , or Groq: https://console.groq.com/keys). Better: store it in Colab *Secrets* (key icon) as `GEMINI_API_KEY` and/or `GROQ_API_KEY`.

In [ ]:
%pip install -q sentence-transformers gradio pytest

In [ ]:
import os, re, json, time, sqlite3, traceback, subprocess, sys, ast, math, hashlib, importlib.util, inspect, shutil, warnings
from contextlib import closing
from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd, requests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

WORK = Path("/content/ai_erp") if Path("/content").exists() else Path("./ai_erp")
WORK.mkdir(parents=True, exist_ok=True)
KNOWLEDGE_DIR = WORK / "knowledge"; KNOWLEDGE_DIR.mkdir(exist_ok=True)   # drop your own policy/SOP .txt/.md files here
GEN_DIR = WORK / "generated";       GEN_DIR.mkdir(exist_ok=True)         # AI-written ERP modules + their tests
DB_SRC_URL = "https://raw.githubusercontent.com/jpwhite3/northwind-SQLite3/main/dist/northwind.db"
DB_PATH = WORK / "erp.db"                                                # working copy (original stays untouched)

def get_secret(name):
    v = os.environ.get(name)
    if v: return v
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

GEMINI_API_KEY = get_secret("GEMINI_API_KEY"); GROQ_API_KEY = get_secret("GROQ_API_KEY")
if not (GEMINI_API_KEY or GROQ_API_KEY):
    from getpass import getpass
    GEMINI_API_KEY = getpass("Free Gemini API key (Enter to skip): ").strip() or None
    if not GEMINI_API_KEY:
        GROQ_API_KEY = getpass("Free Groq API key: ").strip() or None
assert GEMINI_API_KEY or GROQ_API_KEY, "At least one free API key is required."

In [ ]:
# ---------- Provider-agnostic LLM client (free tiers, auto model discovery, automatic fallback) ----------
def extract_json(text):
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.I | re.S)
    try: return json.loads(t)
    except Exception: pass
    for op, cl in (("{", "}"), ("[", "]")):
        i, j = t.find(op), t.rfind(cl)
        if i != -1 and j > i:
            try: return json.loads(t[i:j + 1])
            except Exception: continue
    raise ValueError("No valid JSON in model output")

def extract_code_blocks(text):
    return re.findall(r"```(?:python|py)?[ \t]*\n(.*?)```", text, flags=re.S)

class LLM:
    def __init__(self, gemini_key=None, groq_key=None, min_interval=4.0):
        self.gk, self.qk, self.min_interval = gemini_key, groq_key, min_interval
        self._last, self.calls, self.last_model = 0.0, 0, None
        self.gemini_candidates = [os.environ["GEMINI_MODEL"]] if os.environ.get("GEMINI_MODEL") else (self._list_gemini() if gemini_key else [])
        self.groq_model = os.environ.get("GROQ_MODEL") or (self._pick_groq() if groq_key else None)
        print("Gemini candidates:", self.gemini_candidates[:3], "| Groq model:", self.groq_model)

    def _list_gemini(self):
        try:
            r = requests.get("https://generativelanguage.googleapis.com/v1beta/models", headers={"x-goog-api-key": self.gk}, params={"pageSize": 200}, timeout=30)
            r.raise_for_status()
            names = [m["name"].split("/")[-1] for m in r.json().get("models", []) if "generateContent" in m.get("supportedGenerationMethods", [])]
        except Exception as e:
            print("Gemini model listing failed:", e); return ["gemini-2.0-flash"]
        bad = ("image", "tts", "live", "audio", "embedding", "thinking", "exp", "vision", "robotics", "computer", "learnlm", "gemma", "aqa")
        c = [n for n in names if "flash" in n and not any(b in n for b in bad)]
        ver = lambda n: float(m.group(1)) if (m := re.search(r"gemini-(\d+(?:\.\d+)?)", n)) else 0.0
        c.sort(key=lambda n: (ver(n), "preview" not in n, "lite" not in n, -len(n)), reverse=True)
        return c or names[:3] or ["gemini-2.0-flash"]

    def _pick_groq(self):
        try:
            r = requests.get("https://api.groq.com/openai/v1/models", headers={"Authorization": f"Bearer {self.qk}"}, timeout=30); r.raise_for_status()
            ids = [m["id"] for m in r.json()["data"] if not any(b in m["id"] for b in ("whisper", "guard", "tts", "playai", "orpheus", "distil"))]
            return ([i for i in ids if "llama" in i and "70b" in i] or [i for i in ids if "llama" in i] or [i for i in ids if "gpt-oss" in i] or ids)[0]
        except Exception as e:
            print("Groq model listing failed:", e); return "llama-3.3-70b-versatile"

    def _gemini(self, prompt, system, json_mode, temperature, max_tokens):
        last = "no Gemini candidates"
        for model in self.gemini_candidates[:3]:
            for attempt in range(2):
                body = {"contents": [{"role": "user", "parts": [{"text": prompt}]}], "generationConfig": {"temperature": temperature, "maxOutputTokens": max_tokens}}
                if system: body["systemInstruction"] = {"parts": [{"text": system}]}
                if json_mode: body["generationConfig"]["responseMimeType"] = "application/json"
                try:
                    r = requests.post(f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent", headers={"x-goog-api-key": self.gk, "Content-Type": "application/json"}, json=body, timeout=120)
                except requests.RequestException as e:
                    last = str(e); time.sleep(2); continue
                if r.status_code == 200:
                    cand = (r.json().get("candidates") or [{}])[0]
                    txt = "".join(p.get("text", "") for p in cand.get("content", {}).get("parts", []))
                    if txt.strip(): self.last_model = model; return txt
                    last = f"{model}: empty ({cand.get('finishReason')})"; break
                last = f"{model}: HTTP {r.status_code} {r.text[:150]}"
                if r.status_code >= 500: time.sleep(3 * (attempt + 1)); continue
                if r.status_code == 429: time.sleep(5)
                break   # try next model (separate free quota)
        raise RuntimeError(last)

    def _groq(self, prompt, system, json_mode, temperature, max_tokens):
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
        body = {"model": self.groq_model, "messages": msgs, "temperature": temperature, "max_tokens": min(max_tokens, 8000)}
        if json_mode: body["response_format"] = {"type": "json_object"}
        for attempt in range(3):
            r = requests.post("https://api.groq.com/openai/v1/chat/completions", headers={"Authorization": f"Bearer {self.qk}"}, json=body, timeout=120)
            if r.status_code == 200: self.last_model = self.groq_model; return r.json()["choices"][0]["message"]["content"]
            if r.status_code in (429, 500, 502, 503): time.sleep(5 * (attempt + 1)); continue
            raise RuntimeError(f"groq HTTP {r.status_code} {r.text[:150]}")
        raise RuntimeError("groq retries exhausted")

    def __call__(self, prompt, system="", json_mode=False, temperature=0.2, max_tokens=4096):
        wait = self.min_interval - (time.time() - self._last)
        if wait > 0: time.sleep(wait)
        errs = []
        for name, fn, ok in (("gemini", self._gemini, bool(self.gk)), ("groq", self._groq, bool(self.qk))):
            if not ok: continue
            try:
                out = fn(prompt, system, json_mode, temperature, max_tokens)
                self._last = time.time(); self.calls += 1; return out
            except Exception as e:
                errs.append(f"{name}: {e}")
        raise RuntimeError("All LLM providers failed -> " + " | ".join(errs))

In [ ]:
llm = LLM(GEMINI_API_KEY, GROQ_API_KEY)
print("LLM smoke test:", llm("Reply with the single word OK.", max_tokens=256).strip()[:40], "| model:", llm.last_model)

## 1. Real organization data from GitHub + ERP semantic layer
The raw SQLite file is downloaded from GitHub. A **schema-adaptive semantic layer** (`erp_*` views) gives every AI component one stable vocabulary, so the same code also works with other SQLite ERP-style datasets.

In [ ]:
def download_dataset():
    raw = WORK / "northwind_original.db"
    if not raw.exists():
        r = requests.get(DB_SRC_URL, timeout=180); r.raise_for_status(); raw.write_bytes(r.content)
    with open(raw, "rb") as f: assert f.read(15) == b"SQLite format 3", "Downloaded file is not a SQLite database"
    if not DB_PATH.exists(): shutil.copy(raw, DB_PATH)
    return raw

def con_rw(): return sqlite3.connect(DB_PATH)
def con_ro():
    c = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True); c.execute("PRAGMA query_only=ON"); return c
def q(sql, params=()):
    with closing(con_ro()) as c: return pd.read_sql_query(sql, c, params=params)

download_dataset()
with closing(con_rw()) as c:
    base = [(t, c.execute(f'select count(*) from "{t}"').fetchone()[0]) for (t,) in c.execute("select name from sqlite_master where type='table' and name not like 'sqlite_%'")]
print("Source tables (real GitHub data):"); [print(f"  {t:<28}{n:>8} rows") for t, n in base]

In [ ]:
ENTITIES = {
 "customers":   dict(tables=["Customer", "Customers"], cols={"customer_id": ["Id", "CustomerID", "CustomerId"], "company": ["CompanyName"], "contact": ["ContactName"], "contact_title": ["ContactTitle"], "city": ["City"], "country": ["Country"]}),
 "employees":   dict(tables=["Employee", "Employees"], cols={"employee_id": ["Id", "EmployeeID", "EmployeeId"], "first_name": ["FirstName"], "last_name": ["LastName"], "title": ["Title"], "hire_date": ["HireDate"], "notes": ["Notes"], "reports_to": ["ReportsTo"], "city": ["City"], "country": ["Country"]}),
 "suppliers":   dict(tables=["Supplier", "Suppliers"], cols={"supplier_id": ["Id", "SupplierID", "SupplierId"], "company": ["CompanyName"], "contact": ["ContactName"], "city": ["City"], "country": ["Country"]}),
 "categories":  dict(tables=["Category", "Categories"], cols={"category_id": ["Id", "CategoryID", "CategoryId"], "category_name": ["CategoryName"], "description": ["Description"]}),
 "products":    dict(tables=["Product", "Products"], cols={"product_id": ["Id", "ProductID", "ProductId"], "product_name": ["ProductName"], "supplier_id": ["SupplierId", "SupplierID"], "category_id": ["CategoryId", "CategoryID"], "qty_per_unit": ["QuantityPerUnit"], "unit_price": ["UnitPrice"], "units_in_stock": ["UnitsInStock"], "units_on_order": ["UnitsOnOrder"], "reorder_level": ["ReorderLevel"], "discontinued": ["Discontinued"]}),
 "orders":      dict(tables=["Order", "Orders"], cols={"order_id": ["Id", "OrderID", "OrderId"], "customer_id": ["CustomerId", "CustomerID"], "employee_id": ["EmployeeId", "EmployeeID"], "order_date": ["OrderDate"], "required_date": ["RequiredDate"], "shipped_date": ["ShippedDate"], "ship_via": ["ShipVia"], "freight": ["Freight"], "ship_city": ["ShipCity"], "ship_country": ["ShipCountry"]}),
 "order_lines": dict(tables=["OrderDetail", "Order Details", "OrderDetails"], cols={"order_id": ["OrderId", "OrderID"], "product_id": ["ProductId", "ProductID"], "unit_price": ["UnitPrice"], "quantity": ["Quantity"], "discount": ["Discount"]}),
 "shippers":    dict(tables=["Shipper", "Shippers"], cols={"shipper_id": ["Id", "ShipperID", "ShipperId"], "company": ["CompanyName"]}),
}
VIEW_DESC = {
 "erp_customers": "Customer master data.",
 "erp_employees": "Employee master data (notes = free-text biography).",
 "erp_suppliers": "Supplier master data.",
 "erp_categories": "Product categories.",
 "erp_products": "Product master + inventory position. unit_price = list price; discontinued=1 means discontinued.",
 "erp_orders": "Sales order headers (one row per order).",
 "erp_order_lines": "Sales order lines (one row per product per order).",
 "erp_shippers": "Carriers.",
 "erp_sales_lines": "DENORMALIZED FACT VIEW, one row per order line: order_date (ISO text), order_month ('YYYY-MM'), customer, customer_country, product_name, category_name, supplier_id, quantity, unit_price, discount, revenue = quantity*unit_price*(1-discount). Prefer it for revenue/volume analytics.",
 "erp_order_summary": "One row per order: revenue, freight, days_to_ship, is_late (1 = shipped after required date), shipped_date NULL/empty = still open. Use for delivery performance.",
}
_norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())

def normalize_iso_dates(con, table, col):
    """Make date columns ISO (YYYY-MM-DD) if the source uses another format."""
    sample = [r[0] for r in con.execute(f'select "{col}" from "{table}" where "{col}" is not null and "{col}"<>\'\' limit 50')]
    if not sample or all(re.match(r"^\d{4}-\d{2}-\d{2}", str(v)) for v in sample): return
    d = pd.read_sql_query(f'select rowid as rid, "{col}" as d from "{table}"', con)
    p = pd.to_datetime(d["d"], errors="coerce")
    if p.notna().sum() >= 0.9 * d["d"].notna().sum():
        con.executemany(f'update "{table}" set "{col}"=? where rowid=?', [(x.strftime("%Y-%m-%d"), int(r)) for r, x in zip(d["rid"], p) if pd.notna(x)])

def build_semantic_layer():
    report = {}
    with closing(con_rw()) as c:
        tabs = {_norm(r[0]): r[0] for r in c.execute("select name from sqlite_master where type='table'")}
        for ent, spec in ENTITIES.items():
            t = next((tabs[_norm(x)] for x in spec["tables"] if _norm(x) in tabs), None)
            if not t: report[ent] = "MISSING"; continue
            cols = {_norm(r[1]): r[1] for r in c.execute(f'pragma table_info("{t}")')}
            sel = []
            for canon, cands in spec["cols"].items():
                src = next((cols[_norm(x)] for x in cands if _norm(x) in cols), None)
                if src and canon.endswith("_date") and ent == "orders": normalize_iso_dates(c, t, src)
                sel.append(f'"{src}" AS {canon}' if src else f"NULL AS {canon}")
            c.execute(f"DROP VIEW IF EXISTS erp_{ent}"); c.execute(f'CREATE VIEW erp_{ent} AS SELECT {", ".join(sel)} FROM "{t}"'); report[ent] = t
        c.execute("DROP VIEW IF EXISTS erp_sales_lines")
        c.execute("""CREATE VIEW erp_sales_lines AS
          SELECT ol.order_id, substr(o.order_date,1,10) AS order_date, strftime('%Y-%m', substr(o.order_date,1,10)) AS order_month,
                 o.customer_id, cu.company AS customer, cu.country AS customer_country, o.employee_id,
                 ol.product_id, p.product_name, ca.category_name, p.supplier_id,
                 ol.quantity, ol.unit_price, COALESCE(ol.discount,0) AS discount,
                 ol.quantity*ol.unit_price*(1-COALESCE(ol.discount,0)) AS revenue
          FROM erp_order_lines ol JOIN erp_orders o ON o.order_id=ol.order_id
          LEFT JOIN erp_customers cu ON cu.customer_id=o.customer_id
          LEFT JOIN erp_products p ON p.product_id=ol.product_id
          LEFT JOIN erp_categories ca ON ca.category_id=p.category_id""")
        c.execute("DROP VIEW IF EXISTS erp_order_summary")
        c.execute("""CREATE VIEW erp_order_summary AS
          SELECT o.order_id, substr(o.order_date,1,10) AS order_date, substr(o.required_date,1,10) AS required_date,
                 substr(o.shipped_date,1,10) AS shipped_date, o.customer_id, cu.company AS customer, o.employee_id, o.ship_country, o.freight,
                 SUM(ol.quantity*ol.unit_price*(1-COALESCE(ol.discount,0))) AS revenue,
                 julianday(substr(o.shipped_date,1,10)) - julianday(substr(o.order_date,1,10)) AS days_to_ship,
                 CASE WHEN o.shipped_date IS NOT NULL AND o.shipped_date<>'' AND o.required_date IS NOT NULL AND substr(o.shipped_date,1,10)>substr(o.required_date,1,10) THEN 1 ELSE 0 END AS is_late
          FROM erp_orders o LEFT JOIN erp_order_lines ol ON ol.order_id=o.order_id LEFT JOIN erp_customers cu ON cu.customer_id=o.customer_id
          GROUP BY o.order_id""")
        # ERP write-side tables owned by the AI procurement agent
        c.execute("""CREATE TABLE IF NOT EXISTS ai_purchase_orders(po_id INTEGER PRIMARY KEY AUTOINCREMENT, created_at TEXT, supplier_id TEXT, supplier TEXT,
                     status TEXT, est_value REAL, justification TEXT, email_draft TEXT, lines_json TEXT, approved_by TEXT, approved_at TEXT)""")
        c.execute("CREATE TABLE IF NOT EXISTS ai_audit_log(ts TEXT, actor TEXT, action TEXT, detail TEXT)")
        c.commit()
    return report

LAYER_REPORT = build_semantic_layer()
print("Semantic layer mapping:", LAYER_REPORT)
assert all(LAYER_REPORT[e] != "MISSING" for e in ("customers", "products", "orders", "order_lines")), "Dataset lacks a required ERP entity"
for _e, _t in LAYER_REPORT.items():
    if _t == "MISSING": VIEW_DESC.pop(f"erp_{_e}", None)   # optional entity not present in this dataset

def schema_text():
    out = []
    with closing(con_ro()) as c:
        for v, d in VIEW_DESC.items():
            cols = ", ".join(r[1] for r in c.execute(f"pragma table_info({v})"))
            out.append(f"- {v}({cols}) -- {d}")
    return "\n".join(out)

SCHEMA_TEXT = schema_text()
AS_OF = q("select max(order_date) d from erp_sales_lines")["d"][0]
print(SCHEMA_TEXT); print("\nData as-of date:", AS_OF, "| first order:", q("select min(order_date) d from erp_sales_lines")["d"][0])

## 2. RAG knowledge base built from the organization's own records
Every product, customer, supplier, employee (incl. real free-text notes), category and monthly performance becomes a retrievable document. Your own policy/SOP files (`/content/ai_erp/knowledge/*.txt|md`) are chunked and added too. Multilingual embeddings let staff ask in Persian or English.

In [ ]:
class Embedder:
    def __init__(self, name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        try:
            from sentence_transformers import SentenceTransformer
            self.m = SentenceTransformer(name); self.kind = name
        except Exception as e:
            from sklearn.feature_extraction.text import HashingVectorizer
            self.m = None; self.kind = "hashing-fallback"
            self.h = HashingVectorizer(n_features=2 ** 13, analyzer="char_wb", ngram_range=(3, 5), alternate_sign=False, norm="l2")
            print("WARNING: sentence-transformers unavailable, using lexical fallback:", e)
    def encode(self, texts):
        texts = list(texts)
        if self.m is not None:
            return np.asarray(self.m.encode(texts, batch_size=64, show_progress_bar=False, normalize_embeddings=True), dtype="float32")
        return self.h.transform(texts).astype("float32").toarray()

class VectorIndex:
    """Tiny exact cosine index (numpy). Swap for FAISS/Chroma at >100k documents."""
    def __init__(self, emb): self.emb, self.docs, self.X = emb, [], None
    def add(self, docs):
        if not docs: return
        V = self.emb.encode([d["text"] for d in docs])
        self.X = V if self.X is None else np.vstack([self.X, V]); self.docs += docs
    def search(self, query, k=5, kinds=None):
        if not self.docs: return []
        s = self.X @ self.emb.encode([query])[0]; out = []
        for i in np.argsort(-s):
            d = self.docs[i]
            if kinds and d["kind"] not in kinds: continue
            out.append({**d, "score": float(s[i])})
            if len(out) >= k: break
        return out

def _s(x, default="n/a"):
    return default if x is None or (isinstance(x, float) and np.isnan(x)) or str(x).strip() == "" else str(x).strip()
def _clean(t): return re.sub(r"\s+", " ", t).strip()

def chunk_text(text, size=900, overlap=150):
    paras, chunks, cur = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()], [], ""
    for p in paras:
        if len(cur) + len(p) <= size: cur += ("\n" if cur else "") + p
        else:
            if cur: chunks.append(cur)
            cur = (cur[-overlap:] + "\n" if cur and overlap else "") + p
            while len(cur) > size * 1.5: chunks.append(cur[:size]); cur = cur[size - overlap:]
    if cur: chunks.append(cur)
    return chunks

def knowledge_docs(path):
    path = Path(path); text = path.read_text(errors="ignore")
    return [dict(id=f"policy:{path.stem}#{i}", kind="policy", text=_clean(c), meta={"file": path.name}) for i, c in enumerate(chunk_text(text))]

def build_corpus():
    docs = []
    add = lambda kind, key, text, **meta: docs.append(dict(id=f"{kind}:{key}", kind=kind, text=_clean(text), meta=meta))
    for v, d in VIEW_DESC.items(): add("schema", v, f"{v}: {d}")
    prod = q("""SELECT p.*, s.company AS supplier, s.country AS supplier_country, c.category_name, c.description AS cat_desc,
                COALESCE(x.units,0) units_sold, COALESCE(x.rev,0) revenue, COALESCE(x.n,0) n_orders, x.last_date
                FROM erp_products p LEFT JOIN erp_suppliers s USING(supplier_id) LEFT JOIN erp_categories c USING(category_id)
                LEFT JOIN (SELECT product_id, SUM(quantity) units, SUM(revenue) rev, COUNT(DISTINCT order_id) n, MAX(order_date) last_date FROM erp_sales_lines GROUP BY product_id) x USING(product_id)""")
    for r in prod.itertuples():
        add("product", r.product_id, f"Product {r.product_name} (ID {r.product_id}), category {_s(r.category_name)} ({_s(r.cat_desc)}). Supplier {_s(r.supplier)} ({_s(r.supplier_country)}). Package {_s(r.qty_per_unit)}. List price {_s(r.unit_price)}. Stock {_s(r.units_in_stock)}, on order {_s(r.units_on_order)}, reorder level {_s(r.reorder_level)}. {'DISCONTINUED' if str(r.discontinued) in ('1', 'True') else 'Active'}. Sales history: {r.units_sold:.0f} units, revenue {r.revenue:.0f}, {r.n_orders} orders, last sold {_s(r.last_date)}.")
    cust = q("""SELECT c.*, COALESCE(x.n,0) n_orders, COALESCE(x.rev,0) revenue, x.first_date, x.last_date, x.cats
                FROM erp_customers c LEFT JOIN (SELECT customer_id, COUNT(DISTINCT order_id) n, SUM(revenue) rev, MIN(order_date) first_date, MAX(order_date) last_date,
                GROUP_CONCAT(DISTINCT category_name) cats FROM erp_sales_lines GROUP BY customer_id) x USING(customer_id)""")
    for r in cust.itertuples():
        add("customer", r.customer_id, f"Customer {r.company} (ID {r.customer_id}), contact {_s(r.contact)} ({_s(r.contact_title)}), {_s(r.city)}, {_s(r.country)}. {r.n_orders} orders, revenue {r.revenue:.0f}, first order {_s(r.first_date)}, last order {_s(r.last_date)}. Buys categories: {_s(r.cats)}.")
    emp = q("""SELECT e.*, COALESCE(x.n,0) n_orders, COALESCE(x.rev,0) revenue FROM erp_employees e
               LEFT JOIN (SELECT employee_id, COUNT(DISTINCT order_id) n, SUM(revenue) rev FROM erp_sales_lines GROUP BY employee_id) x USING(employee_id)""")
    for r in emp.itertuples():
        add("employee", r.employee_id, f"Employee {r.first_name} {r.last_name} (ID {r.employee_id}), {_s(r.title)}, hired {_s(r.hire_date)}, based in {_s(r.city)}, {_s(r.country)}, reports to employee {_s(r.reports_to)}. Handled {r.n_orders} orders worth {r.revenue:.0f}. Notes: {_s(r.notes)}")
    sup = q("""SELECT s.*, GROUP_CONCAT(p.product_name, '; ') products FROM erp_suppliers s LEFT JOIN erp_products p USING(supplier_id) GROUP BY s.supplier_id""")
    for r in sup.itertuples():
        add("supplier", r.supplier_id, f"Supplier {r.company} (ID {r.supplier_id}), contact {_s(r.contact)}, {_s(r.city)}, {_s(r.country)}. Supplies: {_s(r.products)}.")
    for r in q("select * from erp_categories").itertuples():
        add("category", r.category_id, f"Category {r.category_name} (ID {r.category_id}): {_s(r.description)}")
    m, top = q("select order_month, sum(revenue) rev, count(distinct order_id) n, count(distinct customer_id) cust from erp_sales_lines group by 1 order by 1"), q("select order_month, product_name, sum(revenue) r from erp_sales_lines group by 1,2")
    for r in m.itertuples():
        t = top[top.order_month == r.order_month].nlargest(3, "r")["product_name"].tolist()
        add("month", r.order_month, f"Month {r.order_month}: revenue {r.rev:.0f} from {r.n} orders and {r.cust} active customers. Top products: {', '.join(t)}.")
    for f in sorted(list(KNOWLEDGE_DIR.glob("*.txt")) + list(KNOWLEDGE_DIR.glob("*.md"))): docs += knowledge_docs(f)
    return docs

EMB = Embedder()
CORPUS = build_corpus()
INDEX = VectorIndex(EMB); INDEX.add(CORPUS)
print(f"RAG index: {len(INDEX.docs)} documents | embedder: {EMB.kind}"); print(pd.Series([d['kind'] for d in INDEX.docs]).value_counts().to_dict())

def add_knowledge_file(path):
    """Personalize the ERP with a new policy/SOP file at runtime (zero training)."""
    path = Path(path); dst = KNOWLEDGE_DIR / path.name
    if path.resolve() != dst.resolve(): shutil.copy(path, dst)
    docs = knowledge_docs(dst); INDEX.add(docs); return len(docs)

## 3. Organization profile (personalization without fine-tuning)
Personalization = (1) auto-generated organization profile injected in every prompt, (2) RAG over own records/documents, (3) self-bootstrapped, execution-verified question→SQL examples retrieved per query. **No model training is ever needed.**

In [ ]:
def org_stats():
    one = lambda sql: q(sql).iloc[0, 0]
    st = dict(customers=int(one("select count(*) from erp_customers")), employees=int(one("select count(*) from erp_employees")), suppliers=int(one("select count(*) from erp_suppliers")),
              products=int(one("select count(*) from erp_products")), orders=int(one("select count(*) from erp_orders")), total_revenue=round(float(one("select sum(revenue) from erp_sales_lines")), 2),
              first_order=str(q("select min(order_date) d from erp_sales_lines").d[0]), last_order=str(AS_OF),
              categories=q("select category_name from erp_categories").category_name.tolist(),
              top_customer_countries=q("select customer_country c, count(distinct order_id) n from erp_sales_lines group by 1 order by 2 desc limit 5").c.tolist())
    return st

STATS = org_stats()
ORG_PROFILE = llm(f"Write a factual 90-word profile of this organization for internal AI assistants (what it trades, scale, geography, period covered, key operational themes). Use ONLY this data:\n{json.dumps(STATS)}", max_tokens=1024).strip()
ORG_SYSTEM = (f"You are the AI assistant inside the ERP of this organization.\nORGANIZATION PROFILE: {ORG_PROFILE}\nDATA AS-OF DATE: {AS_OF} (historical dataset; treat this date as 'today').\n"
              "Always answer in the user's language. Never invent numbers; use only provided data.\n")
INDEX.add([dict(id="org:profile", kind="org", text=_clean(ORG_PROFILE), meta={})])
print(ORG_PROFILE)

## 4. Text-to-SQL agent (read-only, self-repairing, learns from verified examples)

In [ ]:
FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|alter|create|attach|detach|pragma|vacuum|reindex|truncate)\b", re.I)
def run_sql(sql, limit=500):
    s = re.sub(r"/\*.*?\*/", " ", re.sub(r"--[^\n]*", " ", sql), flags=re.S).strip().rstrip(";").strip()
    if ";" in s: raise ValueError("multiple statements are not allowed")
    if not re.match(r"^(select|with)\b", s, re.I): raise ValueError("only SELECT / WITH queries are allowed")
    if FORBIDDEN.search(s): raise ValueError("forbidden keyword in query")
    with closing(con_ro()) as c: return pd.read_sql_query(f"SELECT * FROM ({s}) LIMIT {int(limit)}", c)

EX_INDEX = VectorIndex(EMB)
def add_example(question, sql):
    """Verified Q->SQL pairs (auto-bootstrapped or confirmed by users) are retrieved as few-shot examples."""
    df = run_sql(sql)
    EX_INDEX.add([dict(id=f"ex:{len(EX_INDEX.docs)}", kind="example", text=question, meta={"sql": sql})]); return len(df)

def bootstrap_examples(n=10):
    out = llm(f"Schema (SQLite views):\n{SCHEMA_TEXT}\n\nWrite {n} diverse, genuinely useful business questions that managers of THIS organization would ask (sales, inventory, customers, suppliers, employees, delivery). For each give correct SQLite SQL using only the erp_* views. Return JSON list: [{{\"question\":\"...\",\"sql\":\"...\"}}]", system=ORG_SYSTEM, json_mode=True, max_tokens=6000)
    ok = 0
    for it in extract_json(out):
        try:
            if add_example(it["question"], it["sql"]) > 0: ok += 1
        except Exception: pass
    return ok

def ask_data(question, max_repair=3, narrate=True):
    shots = "\n\n".join(f"Q: {e['text']}\nSQL: {e['meta']['sql']}" for e in EX_INDEX.search(question, k=3))
    system = (ORG_SYSTEM + f"Translate the question into ONE read-only SQLite query over these views:\n{SCHEMA_TEXT}\n"
              "Rules: SELECT/WITH only; use erp_* views; prefer erp_sales_lines for revenue; add LIMIT<=100 for row lists; dates are ISO text; for relative dates use the as-of date, never 'now'.\n"
              f"Verified examples from this organization:\n{shots}\nReturn JSON: {{\"sql\":\"...\",\"assumptions\":\"...\"}}")
    sql = err = df = None; trace = []
    for attempt in range(max_repair + 1):
        prompt = f"Question: {question}" if attempt == 0 else f"Question: {question}\nPrevious SQL:\n{sql}\nProblem: {err}\nReturn a corrected JSON."
        try:
            out = extract_json(llm(prompt, system=system, json_mode=True)); sql = out["sql"]
            df = run_sql(sql)
            if len(df) == 0 and attempt < max_repair: err = "query returned 0 rows; re-check joins, filters and value spellings"; trace.append((sql, err)); df = None; continue
            break
        except Exception as e:
            err = str(e); trace.append((sql, err)); df = None
    if df is None: return dict(ok=False, sql=sql, df=pd.DataFrame(), text=f"Could not produce a valid query: {err}", trace=trace)
    text = ""
    if narrate: text = llm(f"Question: {question}\nSQL result (first rows):\n{df.head(20).to_string(index=False)}\nAnswer in 2-4 sentences in the user's language. Use only numbers present in the result.", system=ORG_SYSTEM, max_tokens=1024).strip()
    return dict(ok=True, sql=sql, df=df, text=text, trace=trace, assumptions=out.get("assumptions", ""))

def rag_answer(question, k=6):
    hits = INDEX.search(question, k=k, kinds={"product", "customer", "employee", "supplier", "category", "month", "org", "policy"})
    ctx = "\n".join(f"[{h['id']}] {h['text']}" for h in hits)
    ans = llm(f"Context:\n{ctx}\n\nQuestion: {question}", system=ORG_SYSTEM + "Answer ONLY from the context and cite sources like [product:12]. If the context lacks the answer, say so.", max_tokens=1500).strip()
    return dict(ok=True, text=ans, sources=[h["id"] for h in hits], df=pd.DataFrame({"source": [h["id"] for h in hits], "score": [round(h["score"], 3) for h in hits]}))

print("Verified examples bootstrapped:", bootstrap_examples(10))

## 5. Operational intelligence: forecasting · replenishment · anomalies · segmentation · delivery risk
Classical, training-free statistical methods (zero GPU cost); the LLM only explains and orchestrates.

In [ ]:
def complete_month_series():
    df = q("select order_month m, sum(revenue) revenue, count(distinct order_id) orders from erp_sales_lines group by 1 order by 1")
    df["m"] = pd.to_datetime(df["m"] + "-01"); df = df.set_index("m").asfreq("MS").fillna(0)
    asof = pd.Timestamp(AS_OF)
    return df.iloc[:-1] if asof.day < asof.days_in_month - 2 else df      # drop a partial last month

def _fit_forecast(y, h):
    y = np.asarray(y, dtype=float)
    try:
        from statsmodels.tsa.holtwinters import ExponentialSmoothing
        if len(y) >= 24: m = ExponentialSmoothing(y, trend="add", damped_trend=True, seasonal="add", seasonal_periods=12)
        elif len(y) >= 6: m = ExponentialSmoothing(y, trend="add", damped_trend=True)
        else: raise ValueError("short series")
        f = m.fit(optimized=True); return np.asarray(f.forecast(h)), float(np.std(f.resid))
    except Exception:
        return np.repeat(y[-3:].mean(), h), float(np.std(y)) if len(y) > 1 else 0.0

def forecast_revenue(horizon=3):
    hist = complete_month_series(); fc, sd = _fit_forecast(hist["revenue"], horizon)
    idx = pd.date_range(hist.index[-1] + pd.offsets.MonthBegin(1), periods=horizon, freq="MS")
    return hist, pd.DataFrame({"month": idx.strftime("%Y-%m"), "forecast": fc.round(0), "lower_80": np.maximum(fc - 1.28 * sd, 0).round(0), "upper_80": (fc + 1.28 * sd).round(0)})

def backtest_forecast(h=3):
    y = complete_month_series()["revenue"].values
    if len(y) < h + 6: return {"note": "series too short for backtest"}
    fc, _ = _fit_forecast(y[:-h], h); naive = np.repeat(y[-h - 1], h); act = y[-h:]
    mape = lambda p: float(np.mean(np.abs(act - p) / np.maximum(act, 1)) * 100)
    return {"holdout_months": h, "MAPE_model_%": round(mape(fc), 1), "MAPE_naive_%": round(mape(naive), 1)}

def reorder_plan(lookback_days=90, lead_days=7, safety_days=7, cover_days=30):
    start = (pd.Timestamp(AS_OF) - pd.Timedelta(days=lookback_days)).strftime("%Y-%m-%d")
    d = q("""SELECT p.product_id, p.product_name, p.supplier_id, s.company AS supplier, p.units_in_stock stock, p.units_on_order on_order, p.reorder_level, p.unit_price, COALESCE(x.u,0) units_lookback
             FROM erp_products p LEFT JOIN erp_suppliers s USING(supplier_id)
             LEFT JOIN (SELECT product_id, SUM(quantity) u FROM erp_sales_lines WHERE order_date>? GROUP BY product_id) x USING(product_id)
             WHERE CAST(COALESCE(p.discontinued,0) AS INTEGER)=0""", params=(start,))
    for c in ["stock", "on_order", "reorder_level", "unit_price", "units_lookback"]: d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)
    d["avg_daily_demand"] = d.units_lookback / lookback_days
    d["position"] = d.stock + d.on_order
    d["reorder_point"] = np.maximum(d.avg_daily_demand * (lead_days + safety_days), d.reorder_level)
    d["days_of_cover"] = np.where(d.avg_daily_demand > 0, d.position / d.avg_daily_demand.where(d.avg_daily_demand > 0, 1), np.inf).round(1)
    d["reorder_flag"] = d.position <= d.reorder_point
    d["suggested_qty"] = np.ceil(np.maximum(0, d.avg_daily_demand * cover_days + d.reorder_point - d.position)).astype(int)
    d["est_value"] = (d.suggested_qty * d.unit_price).round(2)
    d["priority"] = np.select([d.reorder_flag & (d.days_of_cover < lead_days), d.reorder_flag & (d.days_of_cover < lead_days + safety_days)], ["CRITICAL", "HIGH"], default=np.where(d.reorder_flag, "MEDIUM", "OK"))
    return d.sort_values(["reorder_flag", "days_of_cover"], ascending=[False, True]).reset_index(drop=True)

def detect_anomalies(contamination=0.01, top=25):
    from sklearn.ensemble import IsolationForest
    d = q("select order_id, product_id, product_name, customer, order_date, quantity, unit_price, discount, revenue from erp_sales_lines")
    lp = q("select product_id, unit_price list_price from erp_products"); d = d.merge(lp, on="product_id", how="left")
    d["price_ratio"] = (d.unit_price / d.list_price.replace(0, np.nan)).fillna(1.0)
    X = pd.DataFrame({"log_qty": np.log1p(d.quantity), "discount": d.discount, "log_rev": np.log1p(d.revenue), "price_ratio": d.price_ratio})
    iso = IsolationForest(n_estimators=200, contamination=contamination, random_state=42).fit(X)
    d["anomaly_score"] = (-iso.score_samples(X)).round(3); d["flag"] = iso.predict(X) == -1
    mad = 1.4826 * (X - X.median()).abs().median(); scale = mad.where(mad > 1e-6, X.std() + 1e-9)
    z = (X - X.median()) / scale
    d["why"] = [f"unusual {z.columns[np.argmax(np.abs(r))]} (robust z={r[np.argmax(np.abs(r))]:.1f})" for r in z.values]
    return d[d.flag].sort_values("anomaly_score", ascending=False).head(top).drop(columns="flag").reset_index(drop=True)

def delivery_risk():
    d = q("select order_id, customer, order_date, required_date, ship_country, revenue from erp_order_summary where (shipped_date is null or shipped_date='') and required_date is not null")
    d["days_overdue"] = (pd.Timestamp(AS_OF) - pd.to_datetime(d.required_date)).dt.days
    return d[d.days_overdue > 0].sort_values("days_overdue", ascending=False).reset_index(drop=True)

def customer_segments(k=4, explain=False):
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    d = q("select customer_id, customer, max(order_date) last_date, count(distinct order_id) frequency, sum(revenue) monetary from erp_sales_lines group by 1,2")
    d["recency"] = (pd.Timestamp(AS_OF) - pd.to_datetime(d.last_date)).dt.days
    k = min(k, len(d)); X = StandardScaler().fit_transform(np.log1p(d[["recency", "frequency", "monetary"]]))
    d["cluster"] = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    prof = d.groupby("cluster")[["recency", "frequency", "monetary"]].mean()
    score = (-prof.recency.rank() + prof.frequency.rank() + prof.monetary.rank()).sort_values(ascending=False)
    names = ["Champions", "Loyal", "At risk", "Dormant / low value"] if k == 4 else [f"Segment {i + 1}" for i in range(k)]
    label = {c: names[i] for i, c in enumerate(score.index)}; d["segment"] = d.cluster.map(label)
    summ = d.groupby("segment").agg(customers=("customer_id", "count"), avg_recency_days=("recency", "mean"), avg_orders=("frequency", "mean"), avg_revenue=("monetary", "mean")).round(1).sort_values("avg_revenue", ascending=False)
    if explain:
        summ["recommended_action"] = [llm(f"Segment '{i}' with profile {r.to_dict()}. Give ONE concrete CRM action in max 20 words.", system=ORG_SYSTEM, max_tokens=300).strip() for i, r in summ.iterrows()]
    return d.drop(columns="cluster"), summ

def kpi_summary():
    o = q("select count(*) n, sum(revenue) rev, avg(revenue) aov, avg(is_late)*100 late_pct, avg(days_to_ship) ship_days from erp_order_summary").iloc[0]
    rp = reorder_plan()
    return {"orders": int(o.n), "revenue": round(float(o.rev), 0), "avg_order_value": round(float(o.aov), 1), "late_shipments_%": round(float(o.late_pct), 1),
            "avg_days_to_ship": round(float(o.ship_days), 1), "products_to_reorder": int(rp.reorder_flag.sum()), "critical_items": int((rp.priority == "CRITICAL").sum())}

def plot_dashboard():
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    hist, fc = forecast_revenue(3); fig, ax = plt.subplots(2, 2, figsize=(13, 8))
    ax[0, 0].plot(hist.index, hist.revenue, marker="o", label="actual"); ax[0, 0].plot(pd.to_datetime(fc.month + "-01"), fc.forecast, "r--o", label="forecast")
    ax[0, 0].fill_between(pd.to_datetime(fc.month + "-01"), fc.lower_80, fc.upper_80, color="r", alpha=.15); ax[0, 0].set_title("Monthly revenue + 3-month forecast"); ax[0, 0].legend()
    q("select category_name, sum(revenue) r from erp_sales_lines group by 1 order by 2").plot.barh(x="category_name", y="r", ax=ax[0, 1], legend=False); ax[0, 1].set_title("Revenue by category")
    q("select customer, sum(revenue) r from erp_sales_lines group by 1 order by 2 desc limit 10").iloc[::-1].plot.barh(x="customer", y="r", ax=ax[1, 0], legend=False); ax[1, 0].set_title("Top 10 customers")
    rp = reorder_plan(); rp[rp.reorder_flag].priority.value_counts().reindex(["CRITICAL", "HIGH", "MEDIUM"]).fillna(0).plot.bar(ax=ax[1, 1], color=["#c0392b", "#e67e22", "#f1c40f"]); ax[1, 1].set_title("Replenishment alerts by priority")
    fig.tight_layout(); return fig

print("KPIs:", kpi_summary()); print("Forecast backtest:", backtest_forecast())
display(forecast_revenue(3)[1]); display(reorder_plan().query("reorder_flag").head(8)[["product_name", "supplier", "position", "avg_daily_demand", "days_of_cover", "suggested_qty", "priority"]])

## 6. Procurement agent (AI drafts, human approves, everything audited)

In [ ]:
def audit(actor, action, detail):
    with closing(con_rw()) as c: c.execute("insert into ai_audit_log values (?,?,?,?)", (datetime.now().isoformat(timespec="seconds"), actor, action, str(detail)[:2000])); c.commit()

def draft_purchase_orders(max_suppliers=8):
    need = reorder_plan(); need = need[need.reorder_flag & (need.suggested_qty > 0)]
    made = []
    top = need.groupby(["supplier_id", "supplier"]).est_value.sum().sort_values(ascending=False).head(max_suppliers).index
    for sid, sname in top:
        g = need[(need.supplier_id == sid) & (need.supplier == sname)]
        lines = [dict(product_id=int(r.product_id), product=r.product_name, qty=int(r.suggested_qty), priority=r.priority, est_value=float(r.est_value)) for r in g.itertuples()]
        ctx = " ".join(h["text"] for h in INDEX.search(f"supplier {sname}", k=1, kinds={"supplier"}))
        try:
            out = extract_json(llm(f"Supplier context: {ctx}\nProposed purchase lines (JSON): {json.dumps(lines)}\nReturn JSON {{\"justification\":\"max 60 words based on stock/demand\",\"email\":\"short professional purchase request email\"}}", system=ORG_SYSTEM, json_mode=True, max_tokens=1500))
        except Exception as e:
            out = {"justification": f"Automatic replenishment (LLM unavailable: {e})", "email": ""}
        with closing(con_rw()) as c:
            cur = c.execute("insert into ai_purchase_orders(created_at,supplier_id,supplier,status,est_value,justification,email_draft,lines_json) values (?,?,?,?,?,?,?,?)",
                            (datetime.now().isoformat(timespec="seconds"), str(sid), sname, "DRAFT", float(g.est_value.sum()), out.get("justification", ""), out.get("email", ""), json.dumps(lines))); c.commit(); po = cur.lastrowid
        audit("ai-procurement-agent", "PO_DRAFTED", f"PO {po} supplier {sname} value {g.est_value.sum():.0f}"); made.append(po)
    return list_purchase_orders(made)

def list_purchase_orders(ids=None):
    d = q("select po_id, created_at, supplier, status, round(est_value,0) est_value, justification, approved_by from ai_purchase_orders order by po_id desc")
    return d if ids is None else d[d.po_id.isin(ids)]

def approve_purchase_order(po_id, approver):
    """Human-in-the-loop gate: only a named person can approve."""
    if not str(approver).strip(): raise ValueError("approver name required")
    with closing(con_rw()) as c:
        r = c.execute("update ai_purchase_orders set status='APPROVED', approved_by=?, approved_at=? where po_id=? and status='DRAFT'", (approver, datetime.now().isoformat(timespec="seconds"), int(po_id))); c.commit()
    if r.rowcount == 0: return f"PO {po_id} not found or not in DRAFT state."
    audit(approver, "PO_APPROVED", f"PO {po_id}"); return f"PO {po_id} approved by {approver}."

## 7. AI Dev Studio — software development of the ERP itself is AI-driven
A natural-language feature request → LLM writes a **module + pytest tests** (using RAG over the platform API and previously accepted modules) → static security guard → tests run in a subprocess → failures are fed back to the LLM (repair loop, tests may not be weakened) → LLM code review → module registered as a plugin and covered by the regression suite. Humans only review and approve.

In [ ]:
ALLOWED_IMPORTS = {"pandas", "numpy", "math", "datetime", "re", "json", "statistics", "collections", "itertools", "typing", "dataclasses", "sqlite3", "functools"}
TEST_IMPORTS = ALLOWED_IMPORTS | {"pytest", "feature_mod"}
BANNED_CALLS = {"eval", "exec", "compile", "open", "__import__", "input", "globals", "locals", "getattr", "setattr", "delattr", "breakpoint", "vars"}

def static_guard(code, allowed):
    try: tree = ast.parse(code)
    except SyntaxError as e: return [f"SyntaxError: {e}"]
    bad = []
    for n in ast.walk(tree):
        if isinstance(n, ast.Import):
            bad += [f"import of '{a.name}' not allowed" for a in n.names if a.name.split(".")[0] not in allowed]
        elif isinstance(n, ast.ImportFrom):
            if n.level or (n.module or "").split(".")[0] not in allowed: bad.append(f"import from '{n.module}' not allowed")
        elif isinstance(n, ast.Call) and isinstance(n.func, ast.Name) and n.func.id in BANNED_CALLS: bad.append(f"call to '{n.func.id}()' not allowed")
        elif isinstance(n, ast.Attribute) and n.attr.startswith("__") and n.attr not in ("__name__", "__doc__"): bad.append(f"dunder attribute '{n.attr}' not allowed")
    return bad

PLATFORM_API = f"""PLATFORM CONTRACT FOR GENERATED ERP MODULES
A module is a Python file exposing: TITLE (str), DESCRIPTION (str) and run(conn, **params) -> pandas.DataFrame.
`conn` is a READ-ONLY sqlite3 connection supplied by the platform (use pandas.read_sql_query(sql, conn, params=...)). Never open files, never write to the DB.
Allowed imports: {sorted(ALLOWED_IMPORTS)}. Forbidden: os, sys, subprocess, requests, open(), eval/exec/getattr/__import__.
Never use datetime.now(): the dataset is historical. Get 'today' with SELECT MAX(order_date) FROM erp_sales_lines (value: {AS_OF}).
Available SQLite views:
{SCHEMA_TEXT}"""

DEV_INDEX = VectorIndex(EMB); DEV_INDEX.add([dict(id="api:contract", kind="api", text=PLATFORM_API, meta={})])
CONFTEST = '''import os, sys, sqlite3, pytest
sys.path.insert(0, os.path.dirname(__file__))
@pytest.fixture(scope="session")
def conn():
    c = sqlite3.connect(f"file:{os.environ['ERP_DB']}?mode=ro", uri=True); c.execute("PRAGMA query_only=ON"); yield c; c.close()
'''
(GEN_DIR / "conftest.py").write_text(CONFTEST)
REG_FILE = GEN_DIR / "registry.json"
REGISTRY = json.loads(REG_FILE.read_text()) if REG_FILE.exists() else {}
PLUGINS = {}

def run_pytest(workdir, timeout=180):
    env = {**os.environ, "ERP_DB": str(DB_PATH), "PYTHONDONTWRITEBYTECODE": "1"}
    try:
        p = subprocess.run([sys.executable, "-m", "pytest", "-q", "-x", "--tb=short", "-p", "no:cacheprovider", str(workdir)], capture_output=True, text=True, timeout=timeout, cwd=str(workdir), env=env)
        return p.returncode == 0, (p.stdout + p.stderr)[-3500:]
    except subprocess.TimeoutExpired: return False, "pytest timeout"

def load_plugin(slug):
    spec = importlib.util.spec_from_file_location(slug, GEN_DIR / f"{slug}.py"); mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    assert callable(getattr(mod, "run", None)), "module has no run()"; PLUGINS[slug] = mod; return mod

def run_plugin(slug, **params):
    if slug not in PLUGINS: load_plugin(slug)
    with closing(con_ro()) as c: return PLUGINS[slug].run(c, **params)

for _s in list(REGISTRY):
    try: load_plugin(_s)
    except Exception as e: print("could not reload plugin", _s, e)

DEV_SYSTEM = "You are a senior Python/SQL engineer inside an ERP platform. You write small, safe, read-only analytics modules with rigorous pytest tests. Follow the platform contract exactly."
count_tests = lambda t: len(re.findall(r"^def test_", t, re.M))
count_asserts = lambda t: len(re.findall(r"\bassert\b", t))

def build_feature(request, max_iters=4, verbose=True):
    slug = "feat_" + hashlib.sha1(request.encode()).hexdigest()[:8]
    examples = "\n\n".join(f"# existing module for: {d['meta']['request']}\n{d['text'][:1800]}" for d in DEV_INDEX.search(request, k=2, kinds={"plugin"}))
    code = tests = feedback = None; log = []
    for it in range(1, max_iters + 1):
        if it == 1 or code is None:
            prompt = (f"{PLATFORM_API}\n\nAccepted modules for style reference:\n{examples or '(none yet)'}\n\nFEATURE REQUEST: {request}\n\n"
                      "Return EXACTLY two fenced ```python blocks and nothing else:\n1) the module (feature_mod.py)\n2) the pytest file (test_feature_mod.py): `import feature_mod`, use the `conn` fixture, at least 4 tests: "
                      "output is a DataFrame with expected columns; no NaN in key columns; value sanity/ranges; and one cross-check of an aggregate against an independent SQL query written inside the test.")
        else:
            prompt = (f"{PLATFORM_API}\n\nFEATURE REQUEST: {request}\n\nCURRENT MODULE:\n```python\n{code}\n```\nCURRENT TESTS:\n```python\n{tests}\n```\nFEEDBACK:\n{feedback}\n\n"
                      "Fix the problem. Fix the tests ONLY if a test is demonstrably wrong; never delete tests or weaken assertions. Return the same two fenced python blocks.")
        blocks = extract_code_blocks(llm(prompt, system=DEV_SYSTEM, max_tokens=8192))
        if len(blocks) < 2: feedback = "Output must contain exactly two ```python blocks (module, tests)."; log.append((it, "format error")); continue
        new_code, new_tests = blocks[0], blocks[1]
        if tests and (count_tests(new_tests) < count_tests(tests) or count_asserts(new_tests) < count_asserts(tests)): new_tests = tests; log.append((it, "test weakening rejected -> tests restored"))
        code, tests = new_code, new_tests
        problems = static_guard(code, ALLOWED_IMPORTS) + static_guard(tests, TEST_IMPORTS)
        if problems: feedback = "Static security guard violations: " + "; ".join(problems); log.append((it, feedback)); continue
        if count_tests(tests) < 3: feedback = "Provide at least 4 test functions named test_*."; log.append((it, feedback)); continue
        tmp = WORK / "sandbox" / slug; shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
        (tmp / "conftest.py").write_text(CONFTEST); (tmp / "feature_mod.py").write_text(code); (tmp / "test_feature_mod.py").write_text(tests)
        ok, out = run_pytest(tmp); log.append((it, "PASS" if ok else out[-800:]))
        if verbose: print(f"[dev-studio] iteration {it}: {'tests passed' if ok else 'tests failed -> self-repair'}")
        if ok:
            review = llm(f"Review this ERP module in max 5 bullets (correctness, edge cases, performance, SQL safety).\n```python\n{code}\n```", max_tokens=1200).strip()
            (GEN_DIR / f"{slug}.py").write_text(code); (GEN_DIR / f"test_{slug}.py").write_text(tests.replace("feature_mod", slug))
            mod = load_plugin(slug); title = getattr(mod, "TITLE", slug)
            REGISTRY[slug] = dict(title=title, description=getattr(mod, "DESCRIPTION", ""), request=request, iterations=it, created=datetime.now().isoformat(timespec="seconds"), review=review)
            REG_FILE.write_text(json.dumps(REGISTRY, indent=2)); DEV_INDEX.add([dict(id=f"plugin:{slug}", kind="plugin", text=code, meta={"request": request})])
            audit("ai-dev-studio", "MODULE_REGISTERED", f"{slug} after {it} iteration(s)")
            try: preview = run_plugin(slug).head(20)
            except Exception as e: preview = pd.DataFrame({"error": [str(e)]})
            return dict(ok=True, slug=slug, iterations=it, code=code, tests=tests, review=review, preview=preview, log=log)
        feedback = "pytest output:\n" + out
    (GEN_DIR / f"FAILED_{slug}.py").write_text(f"# request: {request}\n{code or ''}")
    return dict(ok=False, slug=slug, iterations=max_iters, code=code or "", tests=tests or "", review="", preview=pd.DataFrame(), log=log)

def core_regression():
    """Deterministic CI gate for the platform itself + every AI-generated module."""
    checks = {}
    checks["views_not_empty"] = all(q(f"select count(*) n from {v}").n[0] > 0 for v in ["erp_customers", "erp_products", "erp_orders", "erp_order_lines", "erp_sales_lines"])
    checks["revenue_reconciles"] = bool(abs(q("select sum(revenue) r from erp_sales_lines").r[0] - q("select sum(revenue) r from erp_order_summary").r[0]) < 1e-6 * max(1, q("select sum(revenue) r from erp_sales_lines").r[0]))
    try: run_sql("DROP TABLE ai_audit_log"); checks["sql_guard_blocks_ddl"] = False
    except ValueError: checks["sql_guard_blocks_ddl"] = True
    checks["reorder_plan_columns"] = {"suggested_qty", "priority"} <= set(reorder_plan().columns)
    checks["forecast_shape"] = len(forecast_revenue(3)[1]) == 3
    if REGISTRY: checks["generated_modules_pass_tests"] = run_pytest(GEN_DIR)[0]
    return checks

print("Core regression:", core_regression())

## 8. ERP Copilot — one natural-language entry point that routes to every AI module

In [ ]:
def narrate(question, df, note=""):
    return llm(f"User request: {question}\n{note}\nTool result (first rows):\n{df.head(15).to_string(index=False)}\nSummarize in 2-4 sentences, in the user's language. Use only numbers present in the result.", system=ORG_SYSTEM, max_tokens=1024).strip()

def copilot(user_msg):
    plugins = {k: v["title"] for k, v in REGISTRY.items()}
    route_sys = ("You are the routing brain of an ERP copilot. Choose exactly ONE tool for the user's message (any language).\nTools:\n"
                 "ask_data: numeric/list questions about ERP records (sales, stock, customers, orders, employees, suppliers).\n"
                 "ask_knowledge: descriptive/qualitative questions about entities, profiles, notes, policies, documents.\n"
                 "forecast: revenue forecast (args: horizon months int).\nreorder: inventory replenishment recommendations.\ndraft_po: create DRAFT purchase orders from replenishment needs.\n"
                 "anomalies: suspicious order lines.\nsegments: customer segmentation.\ndelivery_risk: overdue open orders.\nbuild_feature: user wants a NEW report/module/feature built (args: request).\n"
                 f"run_plugin: run an existing generated module (args: name). Existing plugins: {json.dumps(plugins)}\n"
                 "Return JSON: {\"tool\":\"...\",\"args\":{...}}")
    try: r = extract_json(llm(user_msg, system=route_sys, json_mode=True, max_tokens=300))
    except Exception: r = {"tool": "ask_data", "args": {}}
    tool, args = r.get("tool", "ask_data"), r.get("args") or {}
    if tool == "ask_data": o = ask_data(user_msg); return dict(tool=tool, text=o["text"], df=o["df"], sql=o["sql"])
    if tool == "ask_knowledge": o = rag_answer(user_msg); return dict(tool=tool, text=o["text"], df=o["df"], sql="")
    if tool == "forecast":
        _, fc = forecast_revenue(int(args.get("horizon", 3))); return dict(tool=tool, text=narrate(user_msg, fc, f"Backtest: {backtest_forecast()}"), df=fc, sql="")
    if tool == "reorder":
        d = reorder_plan(); d = d[d.reorder_flag]; return dict(tool=tool, text=narrate(user_msg, d[["product_name", "supplier", "days_of_cover", "suggested_qty", "priority"]]), df=d, sql="")
    if tool == "draft_po": d = draft_purchase_orders(); return dict(tool=tool, text=f"{len(d)} draft purchase orders created. They need human approval.", df=d, sql="")
    if tool == "anomalies": d = detect_anomalies(); return dict(tool=tool, text=narrate(user_msg, d), df=d, sql="")
    if tool == "segments": _, s = customer_segments(explain=False); return dict(tool=tool, text=narrate(user_msg, s.reset_index()), df=s.reset_index(), sql="")
    if tool == "delivery_risk": d = delivery_risk(); return dict(tool=tool, text=narrate(user_msg, d) if len(d) else "No overdue open orders.", df=d, sql="")
    if tool == "build_feature":
        o = build_feature(args.get("request", user_msg)); return dict(tool=tool, text=("Module built and registered: " + o["slug"] if o["ok"] else "Could not build a passing module; see Dev Studio log."), df=o["preview"], sql="")
    if tool == "run_plugin":
        name = args.get("name"); name = name if name in REGISTRY else next((k for k, v in REGISTRY.items() if str(name).lower() in v["title"].lower()), None)
        if not name: return dict(tool=tool, text="No such plugin.", df=pd.DataFrame(), sql="")
        d = run_plugin(name); return dict(tool=tool, text=narrate(user_msg, d), df=d, sql="")
    return dict(tool=tool, text="Unknown tool.", df=pd.DataFrame(), sql="")

## 9. Automatic evaluation (produces the numbers for the report)
Measures: Text-to-SQL execution accuracy (EN + Persian), RAG hit@3 (EN + Persian), forecast backtest, and Dev-Studio success/iterations. Saved to `evaluation_report.json`.

In [ ]:
GOLD = [
 ("What is the total revenue?", "select sum(revenue) from erp_sales_lines"),
 ("مجموع درآمد کل چقدر است؟", "select sum(revenue) from erp_sales_lines"),
 ("Which 5 customers generated the most revenue?", "select customer, sum(revenue) from erp_sales_lines group by customer order by 2 desc limit 5"),
 ("۵ مشتری برتر از نظر درآمد کدامند؟", "select customer, sum(revenue) from erp_sales_lines group by customer order by 2 desc limit 5"),
 ("How many orders were shipped late?", "select sum(is_late) from erp_order_summary"),
 ("Which product category has the highest revenue?", "select category_name from erp_sales_lines group by category_name order by sum(revenue) desc limit 1"),
 ("How many active products are below their reorder level?", "select count(*) from erp_products where cast(coalesce(discontinued,0) as integer)=0 and units_in_stock<reorder_level"),
 ("What is the average number of days to ship an order?", "select avg(days_to_ship) from erp_order_summary"),
 ("Which 3 ship countries have the most orders?", "select ship_country from erp_orders group by ship_country order by count(*) desc limit 3"),
 ("How many distinct customers ordered in the last 90 days of data?", f"select count(distinct customer_id) from erp_sales_lines where order_date > date('{AS_OF}','-90 day')"),
]
def _canon(df):
    cols = []
    for c in df.columns:
        v = df[c].tolist(); cols.append(sorted([round(float(x), 1) if isinstance(x, (int, float, np.integer, np.floating)) and not pd.isna(x) else str(x) for x in v], key=str))
    return cols
def results_match(pred, gold):
    pc = _canon(pred); return all(any(g == p for p in pc) for g in _canon(gold))

def evaluate_all(run_dev=True):
    rep = {"embedder": EMB.kind, "llm": llm.last_model, "as_of": AS_OF}
    rows = []
    for qn, gsql in GOLD:
        try: o = ask_data(qn, narrate=False); ok = o["ok"] and results_match(o["df"], run_sql(gsql))
        except Exception as e: ok = False
        rows.append(dict(question=qn, correct=bool(ok)))
    rep["text2sql"] = {"execution_accuracy": round(float(np.mean([r["correct"] for r in rows])), 3), "n": len(rows), "details": rows}
    prod = q("select product_id, product_name from erp_products").sample(frac=1, random_state=42).head(40)
    hit_en = np.mean([f"product:{r.product_id}" in [h["id"] for h in INDEX.search(f"What are the stock level and supplier of {r.product_name}?", k=3, kinds={"product"})] for r in prod.itertuples()])
    hit_fa = np.mean([f"product:{r.product_id}" in [h["id"] for h in INDEX.search(f"موجودی و تامین‌کننده محصول {r.product_name} چقدر است؟", k=3, kinds={"product"})] for r in prod.itertuples()])
    rep["rag_hit_at_3"] = {"english": round(float(hit_en), 3), "persian": round(float(hit_fa), 3), "n": len(prod)}
    rep["forecast_backtest"] = backtest_forecast(); rep["core_regression"] = core_regression()
    if run_dev:
        reqs = ["Report of customers whose last order is older than 60 days before the as-of date, with days_since_last_order and lifetime revenue, sorted by lifetime revenue.",
                "Monthly revenue by product category with month-over-month growth percentage."]
        res = []
        for r in reqs:
            o = build_feature(r, verbose=False); res.append(dict(request=r, passed=o["ok"], iterations=o["iterations"]))
        rep["dev_studio"] = res
    rep["llm_calls_total"] = llm.calls
    (WORK / "evaluation_report.json").write_text(json.dumps(rep, indent=2, default=str)); return rep

## 10. Smoke-run of every AI module

In [ ]:
for qn in ["Which 5 customers generated the most revenue?", "مجموع درآمد هر دسته‌بندی محصول چقدر است؟"]:
    o = ask_data(qn); print("\nQ:", qn, "\nSQL:", o["sql"], "\nA:", o["text"]); display(o["df"].head())
print("\nRAG:", rag_answer("Tell me about our employee in charge of sales in Europe and her background.")["text"])
print("\nDraft POs:"); display(draft_purchase_orders())
display(detect_anomalies().head(8)); display(customer_segments()[1]); display(delivery_risk().head())

## 11. AI Dev Studio demo — the platform writes and tests a new module by itself

In [ ]:
demo = build_feature("Report listing products whose stock is below reorder level, with columns product_name, supplier, units_in_stock, reorder_level and shortfall (reorder_level - units_in_stock), ordered by shortfall descending. Only active products.")
print("Passed:", demo["ok"], "| iterations:", demo["iterations"]); print(demo["review"]); display(demo["preview"])

## 12. Evaluation (run once; copy the JSON into the report)

In [ ]:
REPORT = evaluate_all(run_dev=True); print(json.dumps(REPORT, indent=2, default=str)[:4000])

## 13. Web UI (free, public Gradio link)

In [ ]:
import gradio as gr

def ui_copilot(msg):
    try: r = copilot(msg)
    except Exception as e: return f"Error: {e}", pd.DataFrame(), ""
    return r["text"], r["df"] if isinstance(r["df"], pd.DataFrame) else pd.DataFrame(), f"tool: {r['tool']}\n{r.get('sql', '')}"
def ui_dashboard(): return plot_dashboard(), json.dumps(kpi_summary(), indent=2)
def ui_reorder(): d = reorder_plan(); return d[d.reorder_flag]
def ui_draft(): return draft_purchase_orders()
def ui_approve(po_id, who):
    try: return approve_purchase_order(po_id, who), list_purchase_orders()
    except Exception as e: return str(e), list_purchase_orders()
def ui_build(req):
    o = build_feature(req, verbose=False)
    return ("PASSED" if o["ok"] else "FAILED") + f" after {o['iterations']} iteration(s)\n\n{o['review']}", o["code"], o["tests"], o["preview"]
def ui_upload(f):
    if f is None: return "No file."
    return f"Indexed {add_knowledge_file(f if isinstance(f, (str, Path)) else f.name)} chunks."
def ui_ask_kb(qn): return rag_answer(qn)["text"]

with gr.Blocks(title="AI-Native ERP") as demo_ui:
    gr.Markdown("# AI-Native ERP  \nReal Northwind data · RAG · Text-to-SQL · forecasting · procurement agent · AI Dev Studio")
    with gr.Tab("Copilot"):
        m = gr.Textbox(label="Ask anything (English / فارسی)"); b = gr.Button("Send"); a = gr.Markdown(); t = gr.Dataframe(); s = gr.Code(label="Details", language="sql")
        b.click(ui_copilot, m, [a, t, s])
    with gr.Tab("Dashboard"):
        b2 = gr.Button("Refresh"); p = gr.Plot(); k = gr.Code(language="json"); b2.click(ui_dashboard, None, [p, k])
    with gr.Tab("Procurement"):
        b3 = gr.Button("Replenishment plan"); t3 = gr.Dataframe(); b3.click(ui_reorder, None, t3)
        b4 = gr.Button("Draft purchase orders (AI)"); t4 = gr.Dataframe(); b4.click(ui_draft, None, t4)
        po = gr.Number(label="PO id", precision=0); who = gr.Textbox(label="Approver name"); b5 = gr.Button("Approve"); msg = gr.Textbox(); t5 = gr.Dataframe(); b5.click(ui_approve, [po, who], [msg, t5])
    with gr.Tab("Dev Studio"):
        r = gr.Textbox(label="Describe a new report/feature", lines=3); b6 = gr.Button("Build, test and register"); st = gr.Textbox(label="Status & AI review", lines=6)
        c1 = gr.Code(label="module", language="python"); c2 = gr.Code(label="tests", language="python"); t6 = gr.Dataframe(); b6.click(ui_build, r, [st, c1, c2, t6])
    with gr.Tab("Knowledge"):
        f = gr.File(label="Upload policy / SOP (.txt, .md)"); o1 = gr.Textbox(); f.upload(ui_upload, f, o1)
        qq = gr.Textbox(label="Ask the knowledge base"); o2 = gr.Markdown(); qq.submit(ui_ask_kb, qq, o2)
demo_ui.queue().launch(share=True, debug=False)